# I. Project Information
Notebook: 01 - Cong Caphe Metadata

# II. Notebook Objectives

# III. Notebook Setup

## III.A. Import Libraries

In [ ]:
import yaml
import json
import warnings
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional
from datetime import datetime
from bs4 import BeautifulSoup

# Python warning ignore
warnings.filterwarnings('ignore')

# Set pandas display options to show all columns and rows
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## III.B. Import Global Config

In [ ]:
# Loading Global Config

PROJECT_ROOT = Path.cwd().parent

def load_config():
    config_path = PROJECT_ROOT / "configs" / "paths.yaml"
    with open(config_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

config = load_config()

## III.C. Import URL

In [ ]:
# Loading Url

url = "https://congcaphe.com"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

In [ ]:
metadata = {
  "title": None,
  "description": None,
  "language": None,
  "keywords": [],
  "robots": None,
  "open_graph": {
    "og_title": None,
    "og_type": None,
    "og_description": None,
    "og_url": None,
    "og_image": None,
    "og_locale": [],
    "og_site_name": None
  },
  "twitter_card": {
    "twitter_card": None,
    "twitter_title": None,
    "twitter_description": None,
    "twitter_image": None,
    "twitter_site": None
  },
  "source_url": url,
  "canonical_url": None,
}

# IV. Cong CaPhe Metadata

## IV.A. Fetch HTML Content

In [ ]:
# Fetch HTML content from the Url

def fetch_html(url: str) -> str:
    response = requests.get(
      url,
      headers=headers,
      verify=False,
      allow_redirects=True
    )
    response.raise_for_status()
    return response.text

html = fetch_html(url)
print(html)

## IV.B. Parse HTML Content

In [ ]:
# Parse HTML content using BeautifulSoup

def parse_html(html: str) -> Optional[BeautifulSoup]:
  try:
    soup = BeautifulSoup(html, "html.parser")
    return soup
  except Exception as e:
    print(f"Error parsing HTML: {e}")
    return None

soup = parse_html(html)
print(soup.prettify())

## IV.C. Extract HTML Content

In [ ]:
# Extract metadata from the parsed HTML content

def extract_metadata(soup: BeautifulSoup, url: str) -> dict:

    # ---- Title ----
    title_tag = soup.find("title")
    if title_tag and title_tag.text:
        metadata["title"] = title_tag.text.strip()

    # ---- Description (name | data-hid | og) ----
    desc_tag = (
        soup.find("meta", attrs={"name": "description"})
        or soup.find("meta", attrs={"data-hid": "description"})
        or soup.find("meta", attrs={"property": "og:description"})
    )
    if desc_tag and desc_tag.get("content"):
        metadata["description"] = desc_tag["content"].strip()

    # ---- Language ----
    language_tag = soup.find("html", lang=True)
    if language_tag and language_tag.get("lang"):
        metadata["language"] = language_tag["lang"].strip()

    # ---- Keywords (list[str]) ----
    keywords_tag = (
        soup.find("meta", attrs={"name": "keywords"})
        or soup.find("meta", attrs={"data-hid": "keywords"})
    )
    if keywords_tag and keywords_tag.get("content"):
        metadata["keywords"] = [
            kw.strip()
            for kw in keywords_tag["content"].split(",")
            if kw.strip()
        ]

    # ---- Robots ----
    robots_tag = (
        soup.find("meta", attrs={"name": "robots"})
        or soup.find("meta", attrs={"data-hid": "robots"})
    )
    if robots_tag and robots_tag.get("content"):
        metadata["robots"] = robots_tag["content"].strip().lower()

    # ---- Canonical URL ----
    canonical_tag = soup.find("link", rel="canonical")
    if canonical_tag and canonical_tag.get("href"):
        metadata["canonical_url"] = canonical_tag["href"].strip()
    else:
        metadata["canonical_url"] = url

    # ---- Open Graph ----
    og_mapping = {
        "og:title": "og_title",
        "og:type": "og_type",
        "og:description": "og_description",
        "og:url": "og_url",
        "og:image": "og_image",
        "og:locale": "og_locale",
        "og:site_name": "og_site_name",
    }

    for og_property, og_field in og_mapping.items():
        og_tag = (
            soup.find("meta", attrs={"property": og_property})
            or soup.find("meta", attrs={"data-hid": og_property})
        )
        if og_tag and og_tag.get("content"):
            if og_field == "og_locale":
                metadata["open_graph"]["og_locale"] = [
                    loc.strip()
                    for loc in og_tag["content"].split(",")
                    if loc.strip()
                ]
            else:
                metadata["open_graph"][og_field] = og_tag["content"].strip()

    # ---- Twitter Card ----
    twitter_mapping = {
        "twitter:card": "twitter_card",
        "twitter:title": "twitter_title",
        "twitter:description": "twitter_description",
        "twitter:image": "twitter_image",
        "twitter:site": "twitter_site",
    }

    for twitter_property, twitter_field in twitter_mapping.items():
        twitter_tag = (
            soup.find("meta", attrs={"name": twitter_property})
            or soup.find("meta", attrs={"property": twitter_property})
            or soup.find("meta", attrs={"data-hid": twitter_property})
        )
        if twitter_tag and twitter_tag.get("content"):
            metadata["twitter_card"][twitter_field] = twitter_tag["content"].strip()

    return metadata

extracted_metadata = extract_metadata(soup, url)
print(json.dumps(extracted_metadata, indent=2, ensure_ascii=False))

## IV.D. Validate HTML Content

In [ ]:
def validate_metadata(metadata: dict) -> dict:

    # -------- Title fallback --------
    if not metadata.get("title"):
        metadata["title"] = metadata.get("open_graph", {}).get("og_title")

    # -------- Description fallback --------
    if not metadata.get("description"):
        metadata["description"] = metadata.get("open_graph", {}).get("og_description")

    # -------- Keywords normalization --------
    keywords = metadata.get("keywords", [])
    if isinstance(keywords, str):
        metadata["keywords"] = [
            kw.strip() for kw in keywords.split(",") if kw.strip()
        ]
    elif keywords is None:
        metadata["keywords"] = []
    else:
        metadata["keywords"] = list(dict.fromkeys(keywords))  # deduplicate, keep order

    # -------- Robots normalization --------
    if metadata.get("robots"):
        metadata["robots"] = metadata["robots"].lower().strip()

    # -------- Language normalization --------
    if metadata.get("language"):
        metadata["language"] = metadata["language"].lower().strip()

    # -------- Canonical URL fallback --------
    if not metadata.get("canonical_url"):
        metadata["canonical_url"] = metadata.get("source_url")

    # -------- Open Graph normalization --------
    og = metadata.get("open_graph", {})

    # og_locale → always list[str]
    og_locale = og.get("og_locale", [])
    if isinstance(og_locale, str):
        og["og_locale"] = [
            loc.strip() for loc in og_locale.split(",") if loc.strip()
        ]
    elif og_locale is None:
        og["og_locale"] = []

    metadata["open_graph"] = og

    # -------- Twitter Card fallback --------
    twitter = metadata.get("twitter_card", {})

    if not twitter.get("twitter_title"):
        twitter["twitter_title"] = metadata.get("title")

    if not twitter.get("twitter_description"):
        twitter["twitter_description"] = metadata.get("description")

    metadata["twitter_card"] = twitter

    # -------- Source domain enrichment (light, useful) --------
    try:
        parsed = urlparse(metadata.get("source_url", ""))
        metadata["source_domain"] = parsed.netloc
    except Exception:
        metadata["source_domain"] = None

    return metadata

validated_metadata = validate_metadata(metadata)
print(json.dumps(validated_metadata, indent=2, ensure_ascii=False))

# V. Export Dataset

## V.A. Export MetaData to Json

In [ ]:
# Export MetaData to Json Format

def export_to_json():
    return

## V.B. Export MetaData to Yaml

In [ ]:
# Export MetaData to Yaml Format

def export_to_yaml():
  return